# B2-022 — Practice p24

*55 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** C · **Type:** challenge · **Difficulty:** advanced · **Time budget:** 55 minutes  
**Concepts:** gaussian-reparameterization

## Task

Audit this VAE pipeline. `TinyVAE` and `negative_elbo` follow the p19 contracts; `candidate_betas = (0.5, 1.0, 2.0)`; `heldout_negative_elbo(beta)` retrains with that $\beta$ and returns the held-out negative ELBO.

```text
 1  data = latent_data.load_rows("mixture8d", range(60))
 2  mean, std = data.mean(dim=0), data.std(dim=0)
 3  data = (data - mean) / std
 4  train = data[list(latent_data.TRAIN_IDS["mixture8d"])]
 5  heldout = data[list(latent_data.HELDOUT_IDS["mixture8d"])]
 6  torch.manual_seed(20260927); model = TinyVAE(8, 16, 2)
 7  optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
 8  for update in range(600):
 9      batch = torch.cat([train, heldout[:4]])
10      optimizer.zero_grad(set_to_none=True)
11      mu, logvar = model.encode(batch)
12      z = torch.distributions.Normal(mu, torch.exp(0.5 * logvar)).sample()
13      x_hat = model.decode(z)
14      loss = negative_elbo(batch, x_hat, mu, logvar)
15      loss.backward(); optimizer.step()
16  best_beta = min(candidate_betas, key=heldout_negative_elbo)
17  report(heldout_negative_elbo(best_beta))      # "final held-out score"
```

Use `TRAIN_IDS["mixture8d"][:40]` as fit rows and `TRAIN_IDS["mixture8d"][40:]` as validation rows in (C); the training batch then has shape `(40,8)`.

(A) **Leakage.** List every train/held-out leakage edge with its line numbers, state which held-out information reaches which training decision, and classify each as data leakage or selection leakage.

(B) **Non-differentiable sampling.** Call `torch.manual_seed(20260927)` immediately before `blocked_sample`. Identify the line that blocks the reconstruction gradient to the encoder. Then demonstrate it in code with the literals in the supplied cell: write `blocked_sample(mu, logvar)` reproducing line 12 and `reparameterize(mu, logvar, eps)` with the p11 contract. Probe that `blocked_sample(mu, logvar).requires_grad` is `False`, and that for `z = reparameterize(mu, logvar, eps)` and `z.sum().backward()`, `mu.grad` is all ones and `logvar.grad` equals `0.5 * torch.exp(0.5 * logvar) * eps` (`ATOL = RTOL = 1e-6`). Explain why the loss on line 14 still decreases during training despite the defect.

(C) **Repair.** Write the repaired pipeline (as code that runs, using the unit loader): normalization statistics from the fit rows only; the training batch is exactly the fit rows; the forward pass uses `model(batch, eps)` with `eps` from a seeded generator; $\beta$ is chosen on the validation rows by the $\beta=1$ negative ELBO with a fixed seeded validation `eps` (comparing $\beta$-weighted losses across different $\beta$ would be biased); held-out rows are evaluated once at the end. You may shorten training to 50 updates for the demonstration.

(D) **Hash check.** Using `latent_data.row_sha256` and `latent_data.ROW_SHA256["mixture8d"]`, assert that none of the raw (pre-normalization) rows your repaired pipeline selects for training or validation is a held-out row, and explain why the hash check must be applied before normalization.

**Required answer-check assertions.** The probes in (B), the hash check in (D), and that the repaired training batch has shape `(40,8)`.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20260927
LOADER_CANDIDATES = (
    Path("../data/latent_data.py"),
    Path("units/B2-022-probabilistic-latent-models/data/latent_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("latent_data", loader_path)
latent_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(latent_data)
torch.set_num_threads(1)

import math

mu = torch.tensor([[0.5, -1.0]], requires_grad=True)
logvar = torch.tensor([[0.0, math.log(0.25)]], requires_grad=True)
eps = torch.tensor([[1.2, -0.4]])
ATOL = 1e-6
RTOL = 1e-6

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.